[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [asyncpg and psycopg3, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/asyncpg-and-psycopg3-deep-dive.html)

# COPY &middot; Solutions


One way to do each task. Not the only way. If yours runs and does what was asked, yours is
right too.

The first cell is the notebook's Setup, with the `loaded` table and fifty thousand rows to write.
Run it first. Each task starts by emptying the table, so they can be run in any order.


In [1]:
import getpass
import io
import os
import subprocess
import sys
import tempfile
import time
from importlib.metadata import PackageNotFoundError, version
from pathlib import Path

try:
    if version("psycopg") < "3.3" or version("asyncpg") < "0.31":
        raise PackageNotFoundError
except PackageNotFoundError:
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "--root-user-action=ignore",
                    "psycopg[binary,pool]==3.3.6", "psycopg-pool==3.3.2", "asyncpg==0.31.0"],
                   check=True)

import asyncpg
import psycopg
from psycopg import errors

def shell(command):
    """Run a shell command and hand back what it printed, without letting it stop the notebook."""
    done = subprocess.run(command, shell=True, capture_output=True, text=True)
    return done.returncode, (done.stdout + done.stderr).strip()


def answering(database="postgres"):
    """Whether a server is there, asked the only way that needs no client binaries."""
    try:
        with psycopg.connect(f"dbname={database}", connect_timeout=2):
            return True
    except psycopg.OperationalError:
        return False


def start_server(wait=60):
    """Install and start PostgreSQL if nothing is answering. Returns what it had to do."""
    if answering():
        return "already running"
    if sys.platform != "linux":
        raise RuntimeError("No PostgreSQL is answering. Start your own server and run this again: "
                           "this cell only installs one on Linux, which is what Colab runs.")

    sudo = "" if os.geteuid() == 0 else "sudo "
    shell(f"{sudo}apt-get -qq update")
    shell(f"{sudo}apt-get -qq -y install postgresql postgresql-contrib")
    shell(f"{sudo}service postgresql start")                        # Colab has no systemd

    for attempt in range(1, wait + 1):                              # start returns before it listens
        if shell("pg_isready -q")[0] == 0:
            break
        print(f"  waiting for the cluster ({attempt})")              # a silent minute looks hung
        time.sleep(1)
    else:
        raise RuntimeError(f"PostgreSQL did not accept connections within {wait} seconds.")

    me = getpass.getuser()                                          # peer authentication wants a role
    asking = f"""sudo -u postgres psql -tAc "SELECT 1 FROM pg_roles WHERE rolname='{me}'" """
    if shell(asking)[1] != "1":                                     # named for the operating system user
        shell(f"sudo -u postgres createuser -s {me}")
    return "installed and started"

def build(rows=5000):
    """Make the guide database and its events table, and fill it once."""
    with psycopg.connect("dbname=postgres", autocommit=True) as conn:
        if not conn.execute("SELECT 1 FROM pg_database WHERE datname = 'guide'").fetchone():
            conn.execute("CREATE DATABASE guide")                   # cannot run in a transaction

    with psycopg.connect("dbname=guide", autocommit=True) as conn:
        for (leftover,) in conn.execute(                            # whatever an earlier run made
                "SELECT tablename FROM pg_tables "
                "WHERE schemaname = 'public' AND tablename <> 'events'").fetchall():
            conn.execute(f'DROP TABLE IF EXISTS "{leftover}" CASCADE')

        conn.execute("""CREATE TABLE IF NOT EXISTS events (
                            id bigserial PRIMARY KEY,
                            ts timestamptz NOT NULL DEFAULT now(),
                            kind text NOT NULL,
                            payload jsonb NOT NULL)""")
        if conn.execute("SELECT count(*) FROM events").fetchone()[0] == 0:
            conn.execute("""INSERT INTO events (kind, payload)
                            SELECT (ARRAY['click', 'view', 'purchase'])[1 + n %% 3],
                                   jsonb_build_object('n', n, 'size', 1 + n %% 7)
                            FROM generate_series(1, %s) AS n""", (rows,))
        return conn.execute("SELECT count(*) FROM events").fetchone()[0]

def report():
    """One line naming what this notebook is running against."""
    rows = build()                                                  # makes the database if it is new
    with psycopg.connect("dbname=guide") as conn:
        major = int(conn.execute("SHOW server_version_num").fetchone()[0]) // 10000
    return (f"PostgreSQL {major} | psycopg {version('psycopg')} | asyncpg {version('asyncpg')} "
            f"| events: {rows} rows")

ROWS = [(n, "click", n % 7) for n in range(1, 50_001)]              # what every load below writes
WORK = Path(tempfile.mkdtemp(prefix="copy-"))


def fresh():
    """An empty table, so each way of loading starts from the same place."""
    with psycopg.connect("dbname=guide", autocommit=True) as conn:
        conn.execute("DROP TABLE IF EXISTS loaded")
        conn.execute("CREATE TABLE loaded (id int, kind text, size int)")


def loaded():
    """How many rows are in it now."""
    with psycopg.connect("dbname=guide") as conn:
        return conn.execute("SELECT count(*) FROM loaded").fetchone()[0]


def timed(load):
    """Seconds taken to run a load against an empty table."""
    fresh()
    start = time.perf_counter()
    load()
    return time.perf_counter() - start


def against(baseline, measured):
    """How much faster, as a band rather than a number.

    A timing on a shared machine is not repeatable to a digit, so this notebook reports which
    band a result fell in. The bands are far enough apart that the answer is the same on every
    run, which a printed ratio was not.
    """
    ratio = baseline / measured
    if ratio < 2:
        return "about the same"
    if ratio <= 10:
        return "several times faster"
    return "an order of magnitude faster"


print("server:", start_server())
print(report())
fresh()
print("loaded is empty:", loaded(), "| rows to write:", len(ROWS))


server: already running
PostgreSQL 16 | psycopg 3.3.6 | asyncpg 0.31.0 | events: 5000 rows
loaded is empty: 0 | rows to write: 50000


**1.** A thousand rows, in one statement.


In [2]:
fresh()

with psycopg.connect("dbname=guide") as conn, conn.cursor() as cur:
    with cur.copy("COPY loaded (id, kind, size) FROM STDIN") as copy:
        for row in ROWS[:1000]:
            copy.write_row(row)

print("rows:", loaded())


rows: 1000


One `COPY`, a thousand `write_row` calls into its stream, and the statement finishes when the block
ends. The server planned one thing rather than a thousand.


**2.** The same rows, written out.


In [3]:
out = WORK / "task-two.csv"

with psycopg.connect("dbname=guide") as conn, conn.cursor() as cur:
    with open(out, "wb") as handle:
        with cur.copy("COPY loaded TO STDOUT WITH (FORMAT csv, HEADER)") as copy:
            for block in copy:
                handle.write(block)

lines = out.read_text().splitlines()
print("first two lines:", lines[:2])
print("lines in all:   ", len(lines), "(a header and the rows)")


first two lines: ['id,kind,size', '1,click,1']
lines in all:    1001 (a header and the rows)


`HEADER` puts the column names on the first line, which is what makes the file readable by anything
else. The source can be a query rather than a table, which is how you export a subset.


**3.** The same rows, in binary.


In [4]:
fresh()

with psycopg.connect("dbname=guide") as conn, conn.cursor() as cur:
    with cur.copy("COPY loaded (id, kind, size) FROM STDIN WITH (FORMAT binary)") as copy:
        copy.set_types(["int4", "text", "int4"])
        for row in ROWS[:1000]:
            copy.write_row(row)

print("rows:", loaded())


rows: 1000


`set_types` is not optional here. A binary stream carries no type information, so the bytes have to
be exactly what the columns expect, and leaving it out gives a `ProtocolViolation` about the message
rather than about the types.


**4.** The two ways, timed.


In [5]:
def by_executemany():
    with psycopg.connect("dbname=guide") as conn, conn.cursor() as cur:
        cur.executemany("INSERT INTO loaded VALUES (%s, %s, %s)", ROWS[:20_000])


def by_copy():
    with psycopg.connect("dbname=guide") as conn, conn.cursor() as cur:
        with cur.copy("COPY loaded (id, kind, size) FROM STDIN") as copy:
            for row in ROWS[:20_000]:
                copy.write_row(row)


baseline = timed(by_executemany)
print("20,000 rows")
print("  executemany: the baseline")
print("  COPY:       ", against(baseline, timed(by_copy)))


20,000 rows
  executemany: the baseline
  COPY:        an order of magnitude faster


The band rather than a number, because the same measurement on the same machine gives a different
digit each time. What does not move is which band it lands in.


**5.** asyncpg's version.


In [6]:
fresh()
conn = await asyncpg.connect(database="guide")

status = await conn.copy_records_to_table("loaded", records=ROWS[:1000],
                                          columns=["id", "kind", "size"])
print("returned:", repr(status))
print("rows:    ", loaded())
await conn.close()


returned: 'COPY 1000'
rows:     1000


The return value is the command tag PostgreSQL sends back, which carries the row count. `COPY 1000`
and a table with a thousand rows in it are the same fact from two directions.


**6.** A file with a bad line.


In [7]:
bad = WORK / "task-six.csv"
bad.write_text("1,click,1\n2,view,2\n3,too,many,columns,here,3\n4,view,4\n")

fresh()
try:
    with psycopg.connect("dbname=guide") as conn, conn.cursor() as cur:
        with cur.copy("COPY loaded (id, kind, size) FROM STDIN WITH (FORMAT csv)") as copy:
            copy.write(bad.read_text())
except errors.BadCopyFileFormat as error:
    print("refused:", error)

print("rows in the table:", loaded(), "<- the two good lines before it are gone too")


refused: extra data after last expected column
CONTEXT:  COPY loaded, line 3: "3,too,many,columns,here,3"
rows in the table: 0 <- the two good lines before it are gone too


A `COPY` is one statement, so it either loads the file or loads none of it. That is usually the right
behavior for a load, and it means the file has to be right before the load is worth starting rather
than being cleaned up afterwards.


---

&#8592; **Back to:** [COPY](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/asyncpg-and-psycopg3-deep-dive/08-copy.ipynb)  &nbsp;&middot;&nbsp;  [asyncpg and psycopg3, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/asyncpg-and-psycopg3-deep-dive.html)
